# Reusable Data Analyst Project Template
### Pattern: Acquire → Clean → Validate → Store → Query → Visualize → Communicate

Use this notebook as a starting skeleton for any project that combines multiple data sources,
cleans them, loads them into SQL, and produces analysis + visuals.

**How to use this template:**
1. Fill in the project framing at the top.
2. Work top-to-bottom, section by section — each section is self-contained.
3. Delete sections you don't need (e.g. skip the API section if you only have one source).
4. Replace placeholder column/table names with your real ones.


## Project Framing

- **Audience / stakeholder:** _who is this for?_
- **Core question(s):** _what decision should this analysis inform?_
- **Data sources:** _list each source and what it contributes_
- **Success criteria:** _what does "done" look like?_


---
## Step 1: Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import requests
import os
import re
from dotenv import load_dotenv

sns.set_theme(style="whitegrid")


---
## Step 2: Data Acquisition

### 2a. Source A — Scrape an HTML table
Use this when the data lives in a `<table>` on a webpage.

In [ ]:
url_a = "PASTE_URL_HERE"

tables = pd.read_html(url_a)
print(f"Found {len(tables)} table(s)")

df_a = tables[0]
df_a.head()


### 2b. Source B — Pull from a REST API
Use this when the data requires authentication and/or pagination.

In [ ]:
load_dotenv()
API_KEY = os.getenv("MY_API_KEY")   # rename to match your .env variable

url_b = "PASTE_API_ENDPOINT_HERE"
params = {"api_key": API_KEY, "limit": 1000, "offset": 0}

n_pages = 10  # adjust based on expected total records / limit
frames = []
errors = []

for i in range(n_pages):
    params["offset"] = i * params["limit"]
    response = requests.get(url_b, params=params)
    if response.status_code == 200:
        payload = response.json()
        # adjust the key below to match the API's actual response structure
        records = payload.get("data", payload)
        frames.append(pd.DataFrame(records))
    else:
        errors.append((params["offset"], response.status_code))

df_b = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Retrieved {len(df_b)} records, {len(errors)} failed requests")
df_b.head()


---
## Step 3: Cleaning

### 3a. Clean Source A

In [ ]:
df_a_clean = df_a.copy()

special_char_pattern = r"[^A-Z0-9 ]"

# --- Text standardization (adjust column name) ---
# df_a_clean["name_col"] = df_a_clean["name_col"].str.upper()
# df_a_clean["name_col"] = df_a_clean["name_col"].str.replace(special_char_pattern, "", regex=True)
# df_a_clean["name_col"] = df_a_clean["name_col"].str.replace("  ", " ")

# --- Dates (adjust column names) ---
# df_a_clean["date_col"] = pd.to_datetime(df_a_clean["date_col"])

# --- Split / parse compound fields ---
# df_a_clean["parsed_col"] = df_a_clean["compound_col"].str.split("/").str[1].str.strip()

df_a_clean.dtypes


### 3b. Clean Source B

In [ ]:
df_b_clean = df_b.copy().drop_duplicates()

# --- Text standardization ---
# df_b_clean["name_col"] = df_b_clean["name_col"].str.upper()
# df_b_clean["name_col"] = df_b_clean["name_col"].str.replace(special_char_pattern, "", regex=True)
# df_b_clean["name_col"] = df_b_clean["name_col"].str.replace("  ", " ")

# --- Map ordinal categories to numeric (example: "$$$" -> 3) ---
# possible_levels = ["$$$$$", "$$$$", "$$$", "$$", "$", " - No ratings yet"]
# for i, level in enumerate(possible_levels):
#     df_b_clean["price_level"] = df_b_clean["price_level"].str.replace(level, str(5 - i))

# --- Extract structured fields with regex ---
# df_b_clean["zip_code"] = df_b_clean["address_col"].str.extract(r"(\d{5})")

# --- Correct dtypes (use nullable Int64 if NaNs are present) ---
# df_b_clean["price_level"] = df_b_clean["price_level"].astype("Int64")
# df_b_clean["zip_code"] = df_b_clean["zip_code"].astype("Int64")

df_b_clean.dtypes


---
## Step 4: Data Quality / Validation

In [ ]:
# Range checks (adjust to your domain, e.g. valid zip codes / dates / scores)
# df_b_clean = df_b_clean[(df_b_clean["zip_code"] >= 10001) & (df_b_clean["zip_code"] <= 11697)]

# Missing value audit
print("Missing values — Source A:")
print(df_a_clean.isna().sum())
print("\nMissing values — Source B:")
print(df_b_clean.isna().sum())


---
## Step 5: Store in SQLite

In [ ]:
db_name = "project_data.db"
connection = sqlite3.connect(db_name)

df_a_clean.to_sql("table_a", connection, if_exists="replace", index=False)
df_b_clean.to_sql("table_b", connection, if_exists="replace", index=False)

print(f"Saved to {db_name}")

# quick sanity check
pd.read_sql_query("SELECT * FROM table_a LIMIT 3", connection)


---
## Step 6: Query & Analyze

### 6a. Group + aggregate

In [ ]:
query_1 = """
SELECT category_col,
       COUNT(*) AS total_records,
       AVG(metric_col) AS avg_metric
FROM table_a
GROUP BY category_col
ORDER BY avg_metric
"""

result_1 = pd.read_sql_query(query_1, connection)
result_1.head()


### 6b. Join two tables

In [ ]:
query_2 = """
SELECT DISTINCT a.col1, a.col2, b.col3
FROM table_a a
JOIN table_b b
  ON a.key_col = b.key_col
WHERE a.col1 IS NOT NULL
ORDER BY a.col2 DESC
"""

result_2 = pd.read_sql_query(query_2, connection)
result_2.head()


### 6c. Conditional aggregation (rate / percentage)

In [ ]:
query_3 = """
SELECT category_col,
       COUNT(*) AS total,
       AVG(CASE WHEN metric_col <= 13 THEN 1 ELSE 0 END) * 100 AS pass_rate,
       AVG(metric_col) AS avg_metric
FROM table_a
GROUP BY category_col
ORDER BY pass_rate DESC
"""

result_3 = pd.read_sql_query(query_3, connection)
result_3.head()


---
## Step 7: Visualize

In [ ]:
# Bar plot — compare a metric across categories
plt.figure(figsize=(8, 5))
sns.barplot(data=result_1.head(15), x="avg_metric", y="category_col")
plt.title("Average Metric by Category")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot — spread across groups
plt.figure(figsize=(8, 5))
sns.boxplot(data=result_2, x="col2", y="col1")
plt.title("Distribution by Group")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Optional: Choropleth map (if your data has a geographic dimension)

In [ ]:
# import geopandas as gpd
#
# shapefile = gpd.read_file("path/to/shapefile.shp")
# shapefile.rename(columns={"ZCTA5CE20": "zip_code"}, inplace=True)
# shapefile["zip_code"] = pd.to_numeric(shapefile["zip_code"])
#
# merged = shapefile.merge(result_1, left_on="zip_code", right_on="category_col", how="left")
# merged.plot(column="avg_metric", cmap="RdBu", legend=True, figsize=(10, 10))
# plt.title("Average Metric by Zip Code")
# plt.show()


---
## Step 8: Close Connection & Summarize Findings

In [ ]:
connection.close()
print("Connection closed.")


### Key Findings
_Summarize 3–5 plain-language insights here for a non-technical stakeholder._

1.
2.
3.
